In [1]:
%%capture
!pip install pip3-autoremove
!pip-autoremove torch torchvision torchaudio -y
!pip install torch torchvision torchaudio xformers --index-url https://download.pytorch.org/whl/cu121
!pip install unsloth

In [2]:
from unsloth import FastLanguageModel, UnslothTrainer, UnslothTrainingArguments
import os
import math
import numpy as np
import torch
from tqdm.auto import tqdm
from datetime import timedelta
import time
import gc

from transformers import (
    DataCollatorForLanguageModeling,
    TrainerCallback,
)
from datasets import load_dataset, Dataset


torch.backends.cudnn.benchmark = True
os.environ["TOKENIZERS_PARALLELISM"] = "false"
torch.cuda.empty_cache()
gc.collect()

# --- Configuration ---
# Using the 16-bit base model
base_model_name = "unsloth/Qwen2.5-Math-1.5B"
# Updated output directory name
output_dir = "./qwen_math_nbody_lora_16bit_from_scratch"
num_epochs_to_train = 3
chunk_size = 2048
validation_size = 600
# *** Increased Learning Rate for initial training ***
unsloth_learning_rate = 1e-4 # More typical starting LR for LoRA

# --- Determine Dtype and 4bit Loading ---
# Unsloth's default dtype=None handles auto-detection, but explicit check is also fine.
if torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8:
    print("GPU supports BF16, using bfloat16.")
    model_dtype = torch.bfloat16
else:
    print("GPU does not support BF16 or is older, using float16.")
    model_dtype = torch.float16

# *** Set load_in_4bit_flag to False for 16-bit training ***
load_in_4bit_flag = False
print(f"Training in 16-bit precision ({model_dtype}). load_in_4bit set to {load_in_4bit_flag}.")

# --- Model Initialization with Unsloth ---
print(f"Initializing Unsloth FastLanguageModel: {base_model_name}...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = base_model_name,
    max_seq_length = chunk_size,
    dtype = model_dtype,
    # *** Pass load_in_4bit=False ***
    load_in_4bit = load_in_4bit_flag,
    # token = "hf_...",
)
print("Base model and tokenizer loaded via Unsloth in 16-bit.")

# --- Apply LoRA using Unsloth's Method ---
# This creates NEW LoRA layers to be trained from scratch
print("Applying NEW LoRA config to model via Unsloth...")
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    lora_alpha = 32,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth", # Still crucial for memory saving in 16-bit!
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)
print("NEW LoRA applied via Unsloth.")
model.print_trainable_parameters() # Verify the number of trainable parameters

# --- REMOVE/COMMENT OUT Adapter Loading Block ---
# print("\nLoading previously trained adapter weights (from book training)...")
# previous_adapter_path = "/kaggle/input/book-adapter-unsloth/pytorch/default/1"
#
# if os.path.exists(previous_adapter_path):
#     try:
#         model.load_adapter(previous_adapter_path, adapter_name="default") # Added adapter_name
#         print(f"Successfully loaded adapter weights from: {previous_adapter_path}")
#     except Exception as e:
#         print(f"Error loading adapter from {previous_adapter_path}: {e}")
#         print("Proceeding with uninitialized LoRA weights (check path and previous run).")
# else:
#     print(f"Warning: Previous adapter path not found: {previous_adapter_path}")
#     print("Proceeding with NEW uninitialized LoRA weights (training from scratch).")
print("\nProceeding with NEW uninitialized LoRA weights (training from scratch).")
print("-" * 40)
# <<< --- END ADAPTER LOADING BLOCK --- >>>

# --- Set Pad Token ---
if tokenizer.pad_token is None:
    print("Setting pad token to eos token.")
    if tokenizer.eos_token is None:
        raise ValueError("Tokenizer does not have an EOS token to use as PAD token.")
    original_vocab_size = len(tokenizer)
    tokenizer.pad_token = tokenizer.eos_token
    if len(tokenizer) > original_vocab_size:
         print(f"Resizing model embeddings from {original_vocab_size} to {len(tokenizer)} due to added pad token...")
         model.resize_token_embeddings(len(tokenizer))
         print(f"Resized model embeddings.")
    else:
         print("Pad token set to existing EOS token. No embedding resize needed.")


# --- Dataset Loading ---
print("Loading dataset...")
# Ensure this path is correct for your NEW training data
data_file_path = '/kaggle/input/book-batch1/cleaned_book_batch1.md' # Assuming this is the correct file
if not os.path.exists(data_file_path):
    raise FileNotFoundError(f"Dataset file not found: {data_file_path}")
try:
    with open(data_file_path, 'r', encoding='utf-8') as f:
        all_lines = f.readlines()
    print(f"Loaded {len(all_lines)} lines from dataset.")
except Exception as e:
    raise IOError(f"Error reading dataset file {data_file_path}: {e}")


# --- Dataset Preparation for Continued Pretraining ---
if not all_lines:
     raise ValueError("Dataset file is empty.")
if len(all_lines) <= validation_size:
     raise ValueError(f"validation_size ({validation_size}) is >= total lines ({len(all_lines)}). Need more data or smaller validation set.")

train_lines = all_lines[:-validation_size]
validation_lines = all_lines[-validation_size:]
print(f"Splitting dataset: {len(train_lines)} training lines, {len(validation_lines)} validation lines.")

train_corpus = "".join(train_lines)
validation_corpus = "".join(validation_lines)

# Function to tokenize and chunk raw text
def prepare_corpus_for_training(corpus, tokenizer, chunk_size, dataset_name=""):
    # (Function remains the same - Ensure it handles padding correctly)
    if not corpus:
        print(f"Warning: Corpus for {dataset_name} is empty. Skipping.")
        return None
    print(f"Tokenizing {dataset_name} corpus...")
    tokens_dict = tokenizer(corpus, truncation=False, add_special_tokens=True, return_attention_mask=False)
    tokens = tokens_dict["input_ids"]
    print(f"Total tokens in {dataset_name}: {len(tokens)}")
    if not tokens:
        print(f"Warning: No tokens generated for {dataset_name}. Check corpus content.")
        return None
    total_expected_chunks = math.ceil(len(tokens) / chunk_size)
    print(f"Creating approximately {total_expected_chunks} chunks of size {chunk_size} for {dataset_name}...")
    chunks = []
    for i in tqdm(range(0, len(tokens), chunk_size), desc=f"Chunking {dataset_name} Data", total=total_expected_chunks):
        chunk_start = i
        chunk_end = i + chunk_size
        chunk_tokens = tokens[chunk_start : chunk_end]
        if len(chunk_tokens) < chunk_size:
            padding_length = chunk_size - len(chunk_tokens)
            if tokenizer.pad_token_id is None:
                 raise ValueError("Tokenizer pad_token_id is None. Cannot pad sequences.")
            chunk_tokens.extend([tokenizer.pad_token_id] * padding_length)
        labels = chunk_tokens[:]
        chunks.append({"input_ids": chunk_tokens, "labels": labels})
    if not chunks:
        print(f"Warning: No chunks were created for {dataset_name}. Check tokenization output and chunk_size.")
        return None
    print(f"Created {len(chunks)} chunks for {dataset_name}.")
    try:
        return Dataset.from_list(chunks)
    except Exception as e:
        print(f"Error creating Dataset object for {dataset_name}: {e}")
        return None


print("Preparing training dataset...")
train_dataset = prepare_corpus_for_training(train_corpus, tokenizer, chunk_size, dataset_name="Training")
if train_dataset:
     print(f"Training dataset size: {len(train_dataset)} chunks")
else:
     raise ValueError("Failed to create training dataset. Check logs.")

print("Preparing validation dataset...")
validation_dataset = prepare_corpus_for_training(validation_corpus, tokenizer, chunk_size, dataset_name="Validation")
if validation_dataset:
    print(f"Validation dataset size: {len(validation_dataset)} chunks")
else:
    print("Warning: Validation dataset could not be created or is empty. Proceeding without evaluation.")
    validation_dataset = None

# --- Data Collator ---
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

# --- Training Arguments using Unsloth ---
print("Defining Unsloth Training Arguments...")
training_args = UnslothTrainingArguments(
    output_dir=output_dir,
    overwrite_output_dir=True,
    num_train_epochs=num_epochs_to_train,
    # *** Reduced batch size & increased accumulation for 16-bit memory ***
    # *** Adjust based on your GPU memory (P100 might need bs=1) ***
    per_device_train_batch_size = 1, # START WITH 1 FOR 16GB GPU and 2k sequence length
    per_device_eval_batch_size = 2,  # Eval can often use slightly more memory
    gradient_accumulation_steps = 32, # Adjust to maintain desired effective batch size (1*32=32 here)
    # *** ***
    learning_rate=unsloth_learning_rate, # Using the increased LR
    weight_decay=0.01,
    logging_steps=10, # Log more frequently initially if needed
    save_strategy="epoch",
    save_total_limit=2,
    dataloader_drop_last=True,
    logging_first_step=True,
    logging_dir=f"{output_dir}/logs",
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    optim="adamw_8bit", # Keep 8-bit optimizer for memory savings on optimizer states
    max_grad_norm=0.5,
    eval_strategy="epoch" if validation_dataset else "no",
    load_best_model_at_end=True if validation_dataset else False,
    metric_for_best_model="eval_loss" if validation_dataset else None,
    greater_is_better=False,
    seed=42,
    report_to=["tensorboard"],
    # resume_from_checkpoint=False, # Ensure this is False or None for training from scratch
)

# --- Trainer Initialization using Unsloth ---
print("Initializing UnslothTrainer...")
trainer = UnslothTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=validation_dataset,
    data_collator=data_collator,
    tokenizer=tokenizer
)

# --- Start Training ---
try:
    num_processes = trainer.accelerator.num_processes
except AttributeError:
    num_processes = 1

effective_batch_size = (
    trainer.args.per_device_train_batch_size *
    trainer.args.gradient_accumulation_steps *
    num_processes
)
print(f"\nEffective batch size: {trainer.args.per_device_train_batch_size} (per_device) * "
      f"{trainer.args.gradient_accumulation_steps} (accumulate) * "
      f"{num_processes} (devices) = {effective_batch_size}")

print(f"Max Sequence Length (chunk_size): {chunk_size}")
print(f"Number of training examples: {len(train_dataset) if train_dataset else 0}")
print(f"Number of validation examples: {len(validation_dataset) if validation_dataset else 0}")
print(f"Number of training epochs: {training_args.num_train_epochs}")
print(f"Optimizer: {training_args.optim}")
print(f"Learning Rate: {training_args.learning_rate}")
print(f"Training Precision: {model_dtype}")
print(f"{'*'*70}\n")


print(f"\n🚀 Starting NEW LoRA training for {trainer.args.num_train_epochs} epochs using Unsloth in 16-bit...\n")
training_start_time = time.time()
try:
    # Ensure resume_from_checkpoint is False or None in args
    train_result = trainer.train(resume_from_checkpoint=training_args.resume_from_checkpoint)
    print("\n✅ Training completed successfully!")

    training_end_time = time.time()
    print(f"Total Training Time: {timedelta(seconds=int(training_end_time - training_start_time))}")

    # --- Save Final Adapter ---
    final_adapter_path = os.path.join(output_dir, "final_adapter")
    print(f"Saving final LoRA adapter to {final_adapter_path}...")
    model.save_pretrained(final_adapter_path)
    tokenizer.save_pretrained(final_adapter_path)
    print(f"Adapter and tokenizer saved successfully to {final_adapter_path}")

    metrics = train_result.metrics
    trainer.log_metrics("train", metrics)
    trainer.save_metrics("train", metrics)
    trainer.save_state()

except KeyboardInterrupt:
    print("\n❌ Training interrupted by user (KeyboardInterrupt).")
    emergency_adapter_path = os.path.join(output_dir, "interrupt_checkpoint_adapter")
    print(f"Attempting to save current adapter state due to interruption...")
    try:
        if hasattr(model, 'save_pretrained'):
             model.save_pretrained(emergency_adapter_path)
             tokenizer.save_pretrained(emergency_adapter_path)
             trainer.save_state()
             print(f"Adapter, tokenizer, and trainer state saved to {emergency_adapter_path}")
        else:
             print("Model object does not support 'save_pretrained'. Cannot save adapter.")
    except Exception as save_e:
        print(f"Failed to save emergency adapter/state: {save_e}")

except Exception as e:
    print(f"\n❌ An error occurred during training: {e}")
    import traceback
    traceback.print_exc() # Print detailed traceback for debugging
    emergency_adapter_path = os.path.join(output_dir, "error_checkpoint_adapter")
    print(f"Attempting to save adapter state due to error...")
    try:
        if hasattr(model, 'save_pretrained'):
             model.save_pretrained(emergency_adapter_path)
             tokenizer.save_pretrained(emergency_adapter_path)
             trainer.save_state()
             print(f"Adapter, tokenizer, and trainer state saved to {emergency_adapter_path}")
        else:
             print("Model object does not support 'save_pretrained'. Cannot save adapter.")
    except Exception as save_e:
        print(f"Failed to save emergency adapter/state after error: {save_e}")

finally:
    print("\nCleaning up GPU memory...")
    del model
    del trainer
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print("\nTraining script finished.")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
GPU does not support BF16 or is older, using float16.
Training in 16-bit precision (torch.float16). load_in_4bit set to False.
Initializing Unsloth FastLanguageModel: unsloth/Qwen2.5-Math-1.5B...
==((====))==  Unsloth 2025.3.19: Fast Qwen2 patching. Transformers: 4.50.2.
   \\   /|    Tesla P100-PCIE-16GB. Num GPUs = 1. Max memory: 15.888 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.5.1+cu121. CUDA: 6.0. CUDA Toolkit: 12.1. Triton: 3.1.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.29.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/165 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/4.87k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

...

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 277 | Num Epochs = 3 | Total steps = 24
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 32
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 32 x 1) = 32
 "-____-"     Trainable parameters = 18,464,768/1,562,179,072 (1.18% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Epoch,Training Loss,Validation Loss
1,1.090700,1.691314
2,1.170600,1.667907


Unsloth: Not an error, but Qwen2ForCausalLM does not accept `num_items_in_batch`.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient



✅ Training completed successfully!
Total Training Time: 0:52:58
Saving final LoRA adapter to ./qwen_math_nbody_lora_16bit_from_scratch/final_adapter...
Adapter and tokenizer saved successfully to ./qwen_math_nbody_lora_16bit_from_scratch/final_adapter
***** train metrics *****
  total_flos               = 11344400GF
  train_loss               =     1.1521
  train_runtime            = 0:52:56.51
  train_samples_per_second =      0.262
  train_steps_per_second   =      0.008

Cleaning up GPU memory...

Training script finished.
